# Stitch Bot — Colab

Apri questo notebook dal telefono (o dal PC) e premi **Runtime → Run all**.
Gira sui server di Google: il telefono serve solo per aprirlo e guardarlo.

**Cosa fa già adesso:** lo *splitter* delle reference + la *roulette* (pesca i
componenti/framer/font dal tuo Drive).

**Cosa manca:** la parte che manda a Stitch (`send_to_stitch.py`) — vedi l'ultima sezione.

## 1) Collega Google Drive
Ti chiederà il permesso: accetta. Serve per leggere reference/arsenal e salvare i risultati.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Installa le librerie

In [ ]:
!pip -q install Pillow

## 3) Splitter delle reference
Imposta **SRC** (cartella reference su Drive) e **OUT** (dove salvare le fette).
Ogni reference finisce in `parte-1-di-3.png`, ecc. La HERO è nella `parte-1`.

In [ ]:
import re
from pathlib import Path
from PIL import Image

# >>> IMPOSTA QUESTI DUE PERCORSI (dentro il tuo Drive) <<<
SRC = '/content/drive/MyDrive/REFENCE'                 # cartella con le reference
OUT = '/content/drive/MyDrive/references_split'        # dove salvare le fette
N   = 3                                                # in quante parti (default 3)

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp'}

def slug(name):
    s = re.sub(r'\.(png|jpe?g|webp)$', '', name, flags=re.I)
    s = re.sub(r'[^a-z0-9]+', '-', s, flags=re.I).strip('-').lower()
    return s[:60] or 'ref'

src, out = Path(SRC), Path(OUT)
assert src.is_dir(), f'Cartella reference non trovata: {src}'
files = sorted(p for p in src.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)
assert files, f'Nessuna immagine in {src}'
out.mkdir(parents=True, exist_ok=True)

for p in files:
    im = Image.open(p).convert('RGB')
    W, H = im.size
    band = -(-H // N)          # ceil(H/N)
    ov = round(H * 0.02)       # 2% overlap ai bordi
    sub = out / slug(p.name)
    sub.mkdir(parents=True, exist_ok=True)
    for old in sub.glob('parte-*.png'):
        old.unlink()
    for i in range(N):
        y0 = max(0, i * band - (ov if i > 0 else 0))
        y1 = min(H, (i + 1) * band + (ov if i < N - 1 else 0))
        im.crop((0, y0, W, y1)).save(sub / f'parte-{i+1}-di-{N}.png', compress_level=1)
    print('ok', slug(p.name), f'({W}x{H}, {N} pezzi)')

print('\nFATTO:', len(files), 'reference divise in', out)

## 4) Roulette — pesca gli ingredienti DAL TUO DRIVE
Imposta **ARSENAL** (la cartella arsenal sul tuo Drive, quella con `componenti`,
`ANIMAZIONI`, `FONT `, ...). La cella elenca i componenti veri dal Drive, esclude
i webgl/three/shader, **preferisce le versioni React** e pesca un giro completo.

In [ ]:
import random
from pathlib import Path

# >>> IMPOSTA la cartella arsenal sul tuo Drive <<<
ARSENAL = '/content/drive/MyDrive/arsenal'   # deve contenere: componenti, ANIMAZIONI, FONT , DESIGN MD, SITI SCARICATI

EXCLUDE = ('three', 'webgl', 'shader')
A = Path(ARSENAL)

def dirs(p):
    p = Path(p)
    return sorted(d.name for d in p.iterdir() if d.is_dir() and not d.name.startswith('.')) if p.is_dir() else []

def is_react(folder):
    if folder.name.lower().endswith('react'):
        return True
    # controllo leggero: cerca un .tsx/.jsx senza scandire tutto
    for f in folder.rglob('*.tsx'):
        return True
    for f in folder.rglob('*.jsx'):
        return True
    return False

comp_dir = A / 'componenti'
comps = [c for c in dirs(comp_dir) if not any(x in c.lower() for x in EXCLUDE)]
react = [c for c in comps if is_react(comp_dir / c)]
pool = react or comps

print('Componenti totali (no webgl):', len(comps), '| di cui React:', len(react))
print('Componente scelto:', random.choice(pool) if pool else 'NESSUNO')

framers = dirs(A / 'ANIMAZIONI' / 'Framer_Interactions')
print('Framer scelti (3):', random.sample(framers, min(3, len(framers))) if framers else 'NESSUNO')

font_parents = dirs(A / 'FONT ')
fonts = [f for f in dirs(A / 'FONT ' / font_parents[0]) if f.lower() != 'static'] if font_parents else []
print('Font scelti (2):', random.sample(fonts, min(2, len(fonts))) if fonts else 'NESSUNO')

> La versione completa (con memoria anti-ripetizione, design DNA e plugin GSAP) è
> in `src/roulette.py` nel repo. Qui la cella è una prova rapida che legge davvero
> dal tuo Drive.

## 5) 🔒 SIGILLATO — invio a Stitch (`send_to_stitch.py`)

La tua automazione, da **NON modificare**. Caricala su Drive intatta, poi imposta la cartella.

> ⚠️ È scritta per **macOS**: su Colab (Linux) l'avvio va adattato a Playwright puro,
> senza toccare la logica di prompt/controllo/invio. Va gestito anche il **login Google**.

In [ ]:
# Prepara il browser per l'invio (serve quando send_to_stitch.py sarà presente)
!pip -q install playwright
!python -m playwright install chromium

In [ ]:
from pathlib import Path

# >>> cartella su Drive dove hai caricato send_to_stitch.py e i suoi file <<<
SENDER_DIR = '/content/drive/MyDrive/reference_splitter'

script = Path(SENDER_DIR) / 'send_to_stitch.py'
if script.exists():
    print('Trovato:', script)
    print('Pronto per il porting/avvio su Colab.')
else:
    print('MANCANTE: send_to_stitch.py non trovato in', SENDER_DIR)
    print('Caricalo su Drive (intatto) e riesegui questa cella.')